# Metric Foundation and Synthetic Validation

Defines the final empty-mask policy, per-bone overlap and physical surface metrics, connected-component agreement, fracture-gap error and false-bridging detection.

This notebook is the authoritative validation surface for Stage 2. It must pass locally before any full HPC decoder run.

In [ ]:
import numpy as np
import pandas as pd
from scipy import ndimage
from scipy.spatial.distance import cdist

BONES = ["femur", "tibia", "patella", "fibula"]
DEFAULT_SPACING = (0.78125, 0.78125, 0.78125)

In [ ]:
def dice_iou(pred, target):
    pred = np.asarray(pred, dtype=bool)
    target = np.asarray(target, dtype=bool)
    intersection = np.logical_and(pred, target).sum()
    pred_n, target_n = pred.sum(), target.sum()
    if pred_n == 0 and target_n == 0:
        return {"dice": 1.0, "iou": 1.0, "empty_prediction": 0}
    if pred_n == 0 and target_n > 0:
        return {"dice": 0.0, "iou": 0.0, "empty_prediction": 1}
    union = np.logical_or(pred, target).sum()
    return {
        "dice": float(2.0 * intersection / (pred_n + target_n)),
        "iou": float(intersection / union),
        "empty_prediction": 0,
    }

def _surface(mask):
    mask = np.asarray(mask, dtype=bool)
    if not mask.any():
        return mask
    return mask ^ ndimage.binary_erosion(mask)

def surface_metrics(pred, target, spacing=DEFAULT_SPACING):
    pred = np.asarray(pred, dtype=bool)
    target = np.asarray(target, dtype=bool)
    spacing = np.asarray(spacing, dtype=float)
    diagonal = float(np.linalg.norm((np.asarray(pred.shape) - 1) * spacing))

    if not pred.any() and not target.any():
        return {"hd95_mm": 0.0, "assd_mm": 0.0, "surface_failure": 0}
    if not pred.any() or not target.any():
        return {"hd95_mm": diagonal, "assd_mm": diagonal, "surface_failure": 1}

    ps, ts = _surface(pred), _surface(target)
    to_target = ndimage.distance_transform_edt(~ts, sampling=spacing)[ps]
    to_pred = ndimage.distance_transform_edt(~ps, sampling=spacing)[ts]
    distances = np.concatenate([to_target, to_pred])
    return {
        "hd95_mm": float(np.percentile(distances, 95)),
        "assd_mm": float(distances.mean()),
        "surface_failure": 0,
    }

In [ ]:
def component_count(mask):
    _, count = ndimage.label(np.asarray(mask, dtype=bool), structure=ndimage.generate_binary_structure(3, 1))
    return int(count)

def false_bridge(pred, target, roi):
    pred = np.asarray(pred, dtype=bool) & np.asarray(roi, dtype=bool)
    target = np.asarray(target, dtype=bool) & np.asarray(roi, dtype=bool)
    gt_labels, gt_count = ndimage.label(target)
    pred_labels, pred_count = ndimage.label(pred)
    if gt_count < 2:
        return {"applicable": 0, "false_bridge": 0, "gt_components": int(gt_count), "pred_components": int(pred_count)}

    connected = False
    for pred_id in range(1, pred_count + 1):
        overlap_ids = np.unique(gt_labels[pred_labels == pred_id])
        overlap_ids = overlap_ids[overlap_ids > 0]
        if len(overlap_ids) >= 2:
            connected = True
            break
    return {
        "applicable": 1,
        "false_bridge": int(connected),
        "gt_components": int(gt_count),
        "pred_components": int(pred_count),
    }

def minimum_component_gap_mm(mask, spacing=DEFAULT_SPACING):
    labels, count = ndimage.label(np.asarray(mask, dtype=bool))
    if count < 2:
        return 0.0
    sizes = ndimage.sum(mask, labels, range(1, count + 1))
    first, second = np.argsort(sizes)[-2:] + 1
    a = np.argwhere(_surface(labels == first)) * np.asarray(spacing)
    b = np.argwhere(_surface(labels == second)) * np.asarray(spacing)
    return float(cdist(a, b).min())

def fracture_gap_error_mm(pred, target, roi, spacing=DEFAULT_SPACING):
    roi = np.asarray(roi, dtype=bool)
    gt_gap = minimum_component_gap_mm(np.asarray(target, dtype=bool) & roi, spacing)
    pred_gap = minimum_component_gap_mm(np.asarray(pred, dtype=bool) & roi, spacing)
    return {
        "target_gap_mm": gt_gap,
        "predicted_gap_mm": pred_gap,
        "gap_error_mm": abs(pred_gap - gt_gap),
    }

In [ ]:
def summarize_subjects(per_sample):
    required = {"subject_id", "dataset", "bone", "dice", "assd_mm", "empty_prediction"}
    missing = required - set(per_sample.columns)
    assert not missing, f"missing columns: {sorted(missing)}"
    per_subject_bone = (
        per_sample.groupby(["subject_id", "dataset", "bone"], as_index=False)
        .agg(dice=("dice", "mean"), assd_mm=("assd_mm", "mean"), empty_prediction=("empty_prediction", "max"))
    )
    return (
        per_subject_bone.groupby(["subject_id", "dataset"], as_index=False)
        .agg(macro_dice=("dice", "mean"), macro_assd_mm=("assd_mm", "mean"), any_empty_prediction=("empty_prediction", "max"))
    )

In [ ]:
shape = (24, 24, 24)
empty = np.zeros(shape, dtype=bool)
cube = np.zeros(shape, dtype=bool)
cube[4:12, 5:13, 6:14] = True

assert dice_iou(empty, cube) == {"dice": 0.0, "iou": 0.0, "empty_prediction": 1}
assert dice_iou(empty, empty) == {"dice": 1.0, "iou": 1.0, "empty_prediction": 0}
assert surface_metrics(empty, cube)["surface_failure"] == 1
assert np.isfinite(surface_metrics(empty, cube)["assd_mm"])

shifted = np.roll(cube, 1, axis=0)
iso = surface_metrics(shifted, cube, spacing=(1.0, 1.0, 1.0))
anisotropic = surface_metrics(shifted, cube, spacing=(2.0, 1.0, 1.0))
assert anisotropic["assd_mm"] > iso["assd_mm"]

fractured = np.zeros(shape, dtype=bool)
fractured[4:10, 7:17, 7:17] = True
fractured[12:18, 7:17, 7:17] = True
roi = np.zeros(shape, dtype=bool)
roi[2:20, 5:19, 5:19] = True
bridged = fractured.copy()
bridged[10:12, 9:15, 9:15] = True

bridge_result = false_bridge(bridged, fractured, roi)
assert bridge_result["applicable"] == 1
assert bridge_result["false_bridge"] == 1
assert false_bridge(fractured, fractured, roi)["false_bridge"] == 0
assert fracture_gap_error_mm(fractured, fractured, roi)["gap_error_mm"] == 0.0
assert fracture_gap_error_mm(bridged, fractured, roi)["gap_error_mm"] > 0.0

print("METRIC FOUNDATION PASS")
print("isotropic shift:", iso)
print("anisotropic shift:", anisotropic)
print("bridge:", bridge_result)

## HPC handoff

The same metric cells must run against one complete 256-cubed validation sample during the decoder preflight. Return the executed notebook or log, metric summary, configuration, resource use and failure traceback. The metric gate remains pending until Agent N records PASS.